In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import scipy.sparse as sp
import pickle

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Colab/job_cleaned.csv')

In [ ]:
df.loc[:,'finalSkill'] = df.loc[:,'finalSkill'].fillna('')

df['combined_text'] = (df['title'] + ' ' +df['finalSkill'] + ' ' +df['finalSkill']) #giving more weightage towards actual skill than role

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

tfidf_vector = vectorizer.fit_transform(df['combined_text'])

print("TF-IDF matrix shape:", tfidf_vector.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

min_exp = df['minimumExperience'].values
max_exp = df['maximumExperience'].values
min_sal = df['minimumSalary'].values
max_sal = df['maximumSalary'].values
salary_disclosed = df['salary_disclosed'].values

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
sp.save_npz('tfidf_matrix.npz', tfidf_vector)
df.to_csv('jobs_model_ready.csv', index=False)



TF-IDF matrix shape: (97622, 5000)
Vocabulary size: 5000


In [ ]:
df = pd.read_csv('jobs_model_ready.csv')

def recommend_jobs(user_skills, user_experience=None, user_expected_salary=None,
                    top_n=10, salary_tolerance=0.15):
    # 1. Similarity between query and every job
    query_vec = vectorizer.transform([user_skills.lower()])
    sims = cosine_similarity(query_vec, tfidf_vector).flatten()

    # 2. Experience filter applies to BOTH lists equally
    exp_mask = np.ones(len(df), dtype=bool)
    if user_experience is not None:
        exp_mask = (df['minimumExperience'] <= user_experience) & \
                   (df['maximumExperience'] >= user_experience)

    # 3. Split the pool in two using salary_disclosed itself
    disclosed_mask = exp_mask & (df['salary_disclosed'] == True)
    undisclosed_mask = exp_mask & (df['salary_disclosed'] == False)

    # 4. Within the disclosed pool, also apply the salary threshold
    if user_expected_salary is not None:
        disclosed_mask &= (df['maximumSalary'] >= user_expected_salary * (1 - salary_tolerance))

    def build_result(mask, cols):
        sims_filtered = np.where(mask, sims, -1)
        top_idx = np.argsort(sims_filtered)[::-1][:top_n]
        # drop rows that never actually matched (score still -1, meaning
        # fewer than top_n jobs passed the filter)
        top_idx = [i for i in top_idx if sims_filtered[i] > -1]
        res = df.iloc[top_idx][cols].copy()
        res['match_score'] = sims_filtered[top_idx]
        return res.reset_index(drop=True)

    disclosed_cols = ['title', 'companyName', 'minimumSalary', 'maximumSalary',
                       'minimumExperience', 'maximumExperience']
    undisclosed_cols = ['title', 'companyName',
                         'minimumExperience', 'maximumExperience']

    disclosed_results = build_result(disclosed_mask, disclosed_cols)
    undisclosed_results = build_result(undisclosed_mask, undisclosed_cols)

    return disclosed_results, undisclosed_results

# ---- test ----
disclosed_df, undisclosed_df = recommend_jobs(
    user_skills="python sql machine learning data analysis",
    user_experience=3,
    user_expected_salary=800000,
    top_n=10
)

print("=== JOBS WITH SALARY INFO ===")
print(disclosed_df.to_string())

print("\n=== JOBS WITHOUT SALARY INFO ===")
print(undisclosed_df.to_string())

=== JOBS WITH SALARY INFO ===
                                        title                         companyName  minimumSalary  maximumSalary  minimumExperience  maximumExperience  match_score
0  machine learning engineer azure databricks                        talentxplore      1200000.0      1700000.0                3.0                8.0     0.541500
1              data scientist junior / senior               de recruitment mumbai      1100000.0      1900000.0                0.0                5.0     0.459276
2              data scientist - ai / ml - mnc                shreesha consultants       800000.0      1800000.0                3.0                8.0     0.402606
3                       sql - python engineer  pr management consultant faridabad      1800000.0      2500000.0                3.0                6.0     0.390838
4                         assistant professor                              uplers       500000.0      1200000.0                0.0                3.0     0